3. Collect weather forecasts from the OpenWeather API.

In [5]:
from pathlib import Path
import pandas as pd
import requests
import time

# Project folders
PROJECT_DIR = Path.cwd().parent
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
#Load my API key
from dotenv import load_dotenv
from pathlib import Path
import os

# Load the .env file from the project root
load_dotenv(Path.cwd().parent / ".env")

API_KEY = os.getenv("OPENWEATHER_API_KEY")

print(API_KEY)

69a13dd57c4f5d0d8f5cc24e00249656


In [8]:
# Load city coordinates
cities_df = pd.read_csv(RAW_DATA_DIR / "cities_coordinates.csv")

cities_df.head()

,city_id,city,latitude,longitude
0,1,Mont Saint Michel,48.635954,-1.511460
1,2,St Malo,48.649518,-2.026041
2,3,Bayeux,49.276462,-0.702474
3,4,Le Havre,49.493898,0.107973
4,5,Rouen,49.440459,1.093966


In [9]:
# Test with the first city
lat = cities_df.loc[0, "latitude"]
lon = cities_df.loc[0, "longitude"]

url = "https://api.openweathermap.org/data/4.0/onecall/timeline/1day"

params = {
    "lat": lat,
    "lon": lon,
    "appid": API_KEY,
    "units": "metric"
}

response = requests.get(url, params=params)

print(response.status_code)

200


In [10]:
# 7-day dataset for all 35 cities
daily_weather_records = []

url = "https://api.openweathermap.org/data/4.0/onecall/timeline/1day"

for _, row in cities_df.iterrows():

    city_id = row["city_id"]
    city = row["city"]
    lat = row["latitude"]
    lon = row["longitude"]

    params = {
        "lat": lat,
        "lon": lon,
        "appid": API_KEY,
        "units": "metric"
    }

    response = requests.get(url, params=params)

    if response.status_code == 200:

        weather_data = response.json()

        # Keep only the next 7 days
        for day in weather_data["data"][:7]:

            daily_weather_records.append({
                "city_id": city_id,
                "city": city,
                "latitude": lat,
                "longitude": lon,
                "date": pd.to_datetime(day["dt"], unit="s"),
                "temp_day": day["temp"]["day"],
                "temp_min": day["temp"]["min"],
                "temp_max": day["temp"]["max"],
                "feels_like_day": day["feels_like"]["day"],
                "humidity": day["humidity"],
                "weather": day["weather"][0]["description"],
                "clouds": day["clouds"],
                "wind_speed": day["wind_speed"],
                "pop": day["pop"],
                "rain": day.get("rain", 0)
            })

        print(f"{city}: OK")

    else:
        print(f"{city}: ERROR {response.status_code}")

Mont Saint Michel: OK
St Malo: OK
Bayeux: OK
Le Havre: OK
Rouen: OK
Paris: OK
Amiens: OK
Lille: OK
Strasbourg: OK
Chateau du Haut Koenigsbourg: OK
Colmar: OK
Eguisheim: OK
Besancon: OK
Dijon: OK
Annecy: OK
Grenoble: OK
Lyon: OK
Gorges du Verdon: OK
Bormes les Mimosas: OK
Cassis: OK
Marseille: OK
Aix en Provence: OK
Avignon: OK
Uzes: OK
Nimes: OK
Aigues Mortes: OK
Saintes Maries de la mer: OK
Collioure: OK
Carcassonne: OK
Ariege: OK
Toulouse: OK
Montauban: OK
Biarritz: OK
Bayonne: OK
La Rochelle: OK


In [11]:
# Create DataFrame

daily_weather_df = pd.DataFrame(daily_weather_records)

daily_weather_df.head()

,city_id,city,latitude,longitude,date,temp_day,temp_min,temp_max,feels_like_day,humidity,weather,clouds,wind_speed,pop,rain
0,1,Mont Saint Michel,48.635954,-1.51146,2026-08-15,24.30,17.98,25.36,24.30,64,light rain,100,6.62,0.8,3.16
1,1,Mont Saint Michel,48.635954,-1.51146,2026-08-16,25.98,16.96,26.20,25.98,55,broken clouds,72,6.87,0.0,0.00
2,1,Mont Saint Michel,48.635954,-1.51146,2026-08-17,22.32,16.11,23.45,22.32,61,overcast clouds,97,5.92,0.0,0.00
3,1,Mont Saint Michel,48.635954,-1.51146,2026-08-18,22.72,16.53,24.38,22.72,66,broken clouds,75,6.08,0.0,0.00
4,1,Mont Saint Michel,48.635954,-1.51146,2026-08-19,20.18,16.18,21.96,20.18,62,overcast clouds,99,4.99,0.0,0.03


In [12]:
# Check data quality

daily_weather_df.shape

(245, 15)

In [13]:
daily_weather_df["city"].nunique()

35

In [14]:
daily_weather_df.isna().sum()

city_id           0
city              0
latitude          0
longitude         0
date              0
temp_day          0
temp_min          0
temp_max          0
feels_like_day    0
humidity          0
weather           0
clouds            0
wind_speed        0
pop               0
rain              0
dtype: int64

In [15]:
daily_weather_df.duplicated().sum()

np.int64(0)

In [ ]:
# Save data

daily_weather_df.to_csv(
    RAW_DATA_DIR / "weather_daily_7days.csv",
    index=False
)